# 05 - Extraction Embeddings LLM (DistilBERT)

**SAE-112** — Epic 3 : Text Representation

## Objectif
Extraire des embeddings de documents à partir de **DistilBERT** pré-entraîné (HuggingFace).
Ces embeddings seront utilisés par les notebooks ML classique (SAE-117) et Deep Learning.

**Grille :** LLM (1pt)

### Stratégie d'extraction
- **Mean Pooling** : moyenne des hidden states de la dernière couche (excluant les tokens spéciaux)
- Dimension de sortie : **768** par document

## 1. Setup & Imports

In [3]:
import sys
import os
sys.path.insert(0, '../..')

import numpy as np
import pandas as pd
import torch
from transformers import DistilBertTokenizer, DistilBertModel
from tqdm import tqdm
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')

# Vérifie le device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {device}')
print(f'PyTorch : {torch.__version__}')
print('Imports OK')

C:\Users\natal\AppData\Roaming\Python\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device : cpu
PyTorch : 2.10.0+cpu
Imports OK


## 2. Chargement des données prétraitées

In [4]:
preprocessed_path = '../../outputs/reviews_preprocessed.pkl'

if os.path.exists(preprocessed_path):
    print(f'Chargement depuis {preprocessed_path}...')
    reviews = pd.read_pickle(preprocessed_path)
    print(f'Chargé {len(reviews):,} reviews')
    print(f'Colonnes : {reviews.columns.tolist()}')
else:
    raise FileNotFoundError(
        f'{preprocessed_path} non trouvé. '
        'Veuillez exécuter le notebook SAE-74 (pipeline preprocessing) d\'abord.'
    )

reviews.head()

FileNotFoundError: ../../outputs/reviews_preprocessed.pkl non trouvé. Veuillez exécuter le notebook SAE-74 (pipeline preprocessing) d'abord.

In [ ]:
# Préparer le texte source pour DistilBERT (texte brut, pas les tokens)
# On utilise la colonne 'text' ou 'text_cleaned' si disponible
if 'text_cleaned' in reviews.columns:
    text_column = 'text_cleaned'
elif 'text' in reviews.columns:
    text_column = 'text'
else:
    # Reconstituer depuis les tokens
    text_column = 'text_reconstructed'
    reviews[text_column] = reviews['tokens_final'].apply(lambda x: ' '.join(x) if isinstance(x, list) else str(x))

print(f'Colonne texte utilisée : {text_column}')
print(f'Exemple : {reviews[text_column].iloc[0][:200]}...')

# Extraire les labels (polarité basée sur les étoiles)
# 1-2 = négatif (0), 3 = neutre (1), 4-5 = positif (2)
if 'stars' in reviews.columns:
    star_col = 'stars'
elif 'stars_review' in reviews.columns:
    star_col = 'stars_review'
else:
    star_col = None
    print('⚠️ Colonne stars non trouvée, les labels ne seront pas sauvegardés')

if star_col:
    reviews['polarity'] = reviews[star_col].apply(
        lambda x: 0 if x <= 2 else (1 if x == 3 else 2)
    )
    print(f'\nDistribution de polarité :')
    print(reviews['polarity'].value_counts().sort_index())
    print('  0=négatif, 1=neutre, 2=positif')

## 3. Chargement de DistilBERT

In [ ]:
model_name = 'distilbert-base-uncased'

print(f'Chargement du tokenizer {model_name}...')
tokenizer = DistilBertTokenizer.from_pretrained(model_name)

print(f'Chargement du modèle {model_name}...')
model = DistilBertModel.from_pretrained(model_name)
model = model.to(device)
model.eval()

print(f'\nModèle chargé !')
print(f'  Paramètres : {sum(p.numel() for p in model.parameters()):,}')
print(f'  Hidden size : {model.config.hidden_size}')
print(f'  Max length  : {tokenizer.model_max_length}')

## 4. Extraction des embeddings

On utilise le **mean pooling** sur les hidden states de la dernière couche,
en masquant les tokens de padding pour ne pas fausser la moyenne.

In [ ]:
def mean_pooling(model_output, attention_mask):
    """
    Mean pooling sur les hidden states, en tenant compte du masque d'attention.
    """
    token_embeddings = model_output.last_hidden_state  # (batch, seq_len, hidden)
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, dim=1)
    sum_mask = torch.clamp(input_mask_expanded.sum(dim=1), min=1e-9)
    return sum_embeddings / sum_mask


def extract_embeddings(texts, tokenizer, model, batch_size=32, max_length=512):
    """
    Extrait les embeddings DistilBERT pour une liste de textes.
    
    Args:
        texts: Liste de textes
        tokenizer: DistilBERT tokenizer
        model: DistilBERT model
        batch_size: Taille des batchs
        max_length: Longueur max de tokenisation (tronqué)
    
    Returns:
        embeddings: np.array de shape (n_texts, 768)
    """
    all_embeddings = []
    
    for i in tqdm(range(0, len(texts), batch_size), desc='Extraction embeddings'):
        batch_texts = texts[i:i + batch_size]
        
        # Tokenisation
        encoded = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors='pt'
        ).to(device)
        
        # Extraction (sans gradient pour économiser la mémoire)
        with torch.no_grad():
            outputs = model(**encoded)
        
        # Mean pooling
        embeddings = mean_pooling(outputs, encoded['attention_mask'])
        all_embeddings.append(embeddings.cpu().numpy())
    
    return np.concatenate(all_embeddings, axis=0)

print('Fonctions d\'extraction définies')

In [ ]:
# Extraire les embeddings
texts = reviews[text_column].fillna('').tolist()

print(f'Extraction des embeddings pour {len(texts):,} documents...')
print(f'Batch size : 32 | Max length : 512')

embeddings = extract_embeddings(texts, tokenizer, model, batch_size=32, max_length=512)

print(f'\nEmbeddings extraits !')
print(f'  Shape : {embeddings.shape}')
print(f'  Dtype : {embeddings.dtype}')
print(f'  Mémoire : {embeddings.nbytes / 1024 / 1024:.1f} MB')

## 5. Vérification rapide

In [ ]:
# Statistiques basiques
print('Statistiques des embeddings :')
print(f'  Moyenne   : {embeddings.mean():.6f}')
print(f'  Std       : {embeddings.std():.6f}')
print(f'  Min       : {embeddings.min():.6f}')
print(f'  Max       : {embeddings.max():.6f}')

# Vérifier les vecteurs nuls
zero_vectors = np.sum(np.all(embeddings == 0, axis=1))
print(f'  Vecteurs nuls : {zero_vectors} / {len(embeddings)}')

# Aperçu du premier embedding
print(f'\nPremier embedding (10 premières dims) :')
print(f'  {embeddings[0][:10]}')

## 6. Visualisation t-SNE

In [ ]:
# Réduction PCA avant t-SNE (accélère le calcul)
print('Réduction PCA (768 → 50)...')
pca = PCA(n_components=50, random_state=42)
embeddings_pca = pca.fit_transform(embeddings)
print(f'Variance expliquée : {pca.explained_variance_ratio_.sum():.2%}')

# t-SNE
print('Calcul t-SNE (50 → 2)...')
tsne = TSNE(n_components=2, random_state=42, perplexity=30, n_iter=1000)
embeddings_2d = tsne.fit_transform(embeddings_pca)

print(f't-SNE terminé : shape = {embeddings_2d.shape}')

In [ ]:
# Visualisation
fig, ax = plt.subplots(figsize=(12, 8))

if 'polarity' in reviews.columns:
    colors = reviews['polarity'].values
    scatter = ax.scatter(
        embeddings_2d[:, 0], embeddings_2d[:, 1],
        c=colors, cmap='RdYlGn', alpha=0.5, s=10
    )
    cbar = plt.colorbar(scatter, ax=ax)
    cbar.set_ticks([0, 1, 2])
    cbar.set_ticklabels(['Négatif', 'Neutre', 'Positif'])
else:
    ax.scatter(
        embeddings_2d[:, 0], embeddings_2d[:, 1],
        alpha=0.5, s=10, color='steelblue'
    )

ax.set_title('t-SNE des embeddings DistilBERT', fontsize=14, fontweight='bold')
ax.set_xlabel('t-SNE 1')
ax.set_ylabel('t-SNE 2')

plt.tight_layout()
os.makedirs('../../outputs/figures', exist_ok=True)
plt.savefig('../../outputs/figures/tsne_distilbert.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure sauvegardée : outputs/figures/tsne_distilbert.png')

## 7. Sauvegarde des embeddings

In [ ]:
# Sauvegarder dans data/ pour les notebooks ML
output_dir = '../../data'
os.makedirs(output_dir, exist_ok=True)

embeddings_path = os.path.join(output_dir, 'distilbert_embeddings.npy')
np.save(embeddings_path, embeddings)
print(f'Embeddings sauvegardés : {embeddings_path}')
print(f'  Shape : {embeddings.shape}')

# Sauvegarder les labels
if 'polarity' in reviews.columns:
    labels = reviews['polarity'].values
    labels_path = os.path.join(output_dir, 'distilbert_labels.npy')
    np.save(labels_path, labels)
    print(f'Labels sauvegardés   : {labels_path}')
    print(f'  Shape : {labels.shape}')
    print(f'  Distribution : {dict(zip(*np.unique(labels, return_counts=True)))}')

print(f'\n✅ Fichiers prêts pour SAE-117 (ML sur embeddings LLM)')

In [ ]:
print('\n✅ Notebook SAE-112 terminé avec succès !')
print(f'   Embeddings DistilBERT extraits pour {len(embeddings):,} documents')
print(f'   Dimension : {embeddings.shape[1]}')
print(f'   Fichiers : data/distilbert_embeddings.npy, data/distilbert_labels.npy')